# 5.1.2 Model 2: Random Forest Regressor

Trained on `X_train.csv` (unscaled) since tree-based models don't need feature scaling.

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
import sys, os
sys.path.append(os.path.dirname(os.path.abspath('__file__')))
from model_utils import evaluate_model, cross_validate_model

MODELLING_DIR = os.path.join("..", "data", "modelling")

X_train = pd.read_csv(os.path.join(MODELLING_DIR, "X_train.csv"))
X_test = pd.read_csv(os.path.join(MODELLING_DIR, "X_test.csv"))
y_train = pd.read_csv(os.path.join(MODELLING_DIR, "y_train.csv"))["price"]
y_test = pd.read_csv(os.path.join(MODELLING_DIR, "y_test.csv"))["price"]

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")


X_train: (3004, 51) | X_test: (751, 51)


## Hyperparameter Tuning
A small grid searched with 5-fold CV on the training set only (`GridSearchCV` scores on log(price) internally for ranking configurations - this is fine for *comparing* configurations against each other, since a lower log-scale error consistently means a lower RM-scale error too; the final chosen model is then re-evaluated in RM using `evaluate_model`).

In [2]:
param_grid = {
    "n_estimators": [200, 400],
    "max_depth": [None, 15],
    "min_samples_leaf": [1, 3],
}

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
print(f"Best CV score (log-scale RMSE): {-grid_search.best_score_:.4f}")


Best params: {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 400}
Best CV score (log-scale RMSE): 0.2651


## Train Final Model

In [3]:
model = RandomForestRegressor(**grid_search.best_params_, random_state=42)
model.fit(X_train, y_train)
print("Model trained.")


Model trained.


## Evaluate
Metrics computed on both train and test sets for Section 6.2's overfitting/underfitting analysis.

In [4]:
train_metrics = evaluate_model(model, X_train, y_train, label="Train")
print()
test_metrics = evaluate_model(model, X_test, y_test, label="Test")


Train RMSE:  RM 70,558  (20.2% of median price)
Train MAE:   RM 29,522
Train MAPE:  6.7%
Train R2:    0.9537
Train MSE:   4,978,424,903



Test RMSE:  RM 179,502  (49.9% of median price)
Test MAE:   RM 78,478
Test MAPE:  16.7%
Test R2:    0.7073
Test MSE:   32,220,899,389


## 5-fold Cross-Validation
Run on X_train only (X_test stays untouched), using the tuned hyperparameters.

In [5]:
cv_results = cross_validate_model(
    RandomForestRegressor(**grid_search.best_params_, random_state=42),
    X_train, y_train, n_splits=5,
)


5-fold CV (mean +/- std):
  RMSE:  RM 167,346 +/- 24,801  (47.6% of median price)
  MAE:   RM 77,709 +/- 3,172
  MAPE:  19.2% +/- 1.5%
  R2:    0.7287 +/- 0.0687
  MSE:   28,619,732,268 +/- 9,046,393,232


## Feature Importance vs EDA (Section 4.5.2)
Section 4.5.2 ranked Property Size, Bathroom, Parking Lot, and the Has_Gymnasium/Has_Swimming_Pool amenities as the strongest numerical correlates of price, and separately found Property Type (eta-squared 0.424) and State (eta-squared 0.172) to be strong categorical predictors. This checks whether Random Forest's learned importances broadly agree.

In [6]:
importance_table = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print("Top 15 features by importance:")
print(importance_table.head(15))


Top 15 features by importance:
Property Size                     0.540794
Has_Gymnasium                     0.054721
State_Penang                      0.049239
Has_Security                      0.030691
Property Age                      0.027883
# of Floors                       0.025643
PropertyType_Flat                 0.024500
State_Selangor                    0.024345
Total Units                       0.021238
PropertyType_Service_Residence    0.015670
Listed_Facility_Count             0.015479
Bathroom                          0.012981
Parking Lot                       0.012970
PropertyType_Condominium          0.012044
Bedroom                           0.011277
dtype: float64


## Save Trained Model
Saved for the Streamlit prototype (Section 8) to load directly, without retraining.

In [7]:
import joblib
MODEL_DIR = os.path.join("..", "models")
os.makedirs(MODEL_DIR, exist_ok=True)
model_path = os.path.join(MODEL_DIR, "random_forest_model.pkl")
joblib.dump(model, model_path)
print(f"Model saved to {model_path}")

Model saved to ..\models\random_forest_model.pkl
